* Had to check is tyre defective or not.

* By using pretrained ResNet 50, in this i had to remove last fully connected layer (which had 10 class labels) and add our fully connected layer (with 2 class label) to use this to train out model to check tyre is defective or not. actually i need to do classification.

* 

## 1. Importing Libraries & Checking the Environment

**Why:** Load every library the demo uses, and confirm PyTorch works in this notebook.
- `torch` — core deep-learning library (tensors, autograd, optimizers).
- `torch.nn` — building blocks: layers, losses, activations.
- `torchvision.models` — ready-made architectures (ResNet50) + their pretrained weights.
- `torchvision.transforms` — image preprocessing pipeline.
- `ImageFolder` — reads images from subfolders and auto-assigns class labels.
- `DataLoader / Subset / random_split` — batching, subsetting, train/test splitting.

**Cross-questions:**
- *Q: Why torchvision instead of OpenCV for loading images?*
  A: torchvision's `ToTensor` converts PIL images directly into PyTorch tensors in the
  [C,H,W] layout the model expects, and ships ImageNet normalization statistics.
- *Q: Why check `torch.cuda.is_available()` first?*
  A: Every tensor and model must live on the same device; we decide it once here.

In [1]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset, random_split

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())   # False on this machine -> CPU training

PyTorch version: 2.10.0+cu128
GPU available: False


In [2]:
# Folder that CONTAINS the "defective/" and "good/" subfolders
DATASET_ROOT = "/home/ayush-ubantu/Downloads/Jamia/Lab/Sem3/MS36/tyre_images"

# Quick-test mode: images to use from EACH class.
# 150 -> fast smoke test (~2-3 min).  None -> full dataset (1856 images) for the final run.
N_PER_CLASS = 150

In [3]:
# Preprocessing: make our tyre images numerically similar to the ImageNet images
# the pretrained model learned from.
preprocess = transforms.Compose([
    transforms.Resize(256),      # shorter side -> 256px, aspect ratio preserved
    transforms.CenterCrop(224),  # central 224x224 crop -> ResNet50's required input size
    transforms.ToTensor(),       # PIL image -> tensor [C,H,W], pixels scaled to [0, 1]
    transforms.Normalize(mean=[0.485, 0.456, 0.406],     # ImageNet per-channel mean (R, G, B)
                         std=[0.229, 0.224, 0.225])      # ImageNet per-channel std
])

In [4]:
# Load all images; subfolder names become the class labels (alphabetical order)
dataset = ImageFolder(root=DATASET_ROOT, transform=preprocess)

print("Classes (alphabetical order):", dataset.classes)    # ['defective', 'good']
print("Class to index:", dataset.class_to_idx)             # {'defective': 0, 'good': 1}
print("Total images found:", len(dataset))                 # 1856 in the full dataset

# ---- Quick-test subset (only active when N_PER_CLASS is set) ----
if N_PER_CLASS is not None:
    indices, counts = [], {}
    # Walk the file list (no images loaded) and keep the first N_PER_CLASS of each class
    for idx, (path, label) in enumerate(dataset.samples):
        if counts.get(label, 0) < N_PER_CLASS:
            indices.append(idx)
            counts[label] = counts.get(label, 0) + 1
    dataset = Subset(dataset, indices)
    print("Smoke-test images per class index:", counts)    # e.g. {0: 150, 1: 150}
else:
    print("Using the FULL dataset")

# Peek at one sample: the transform already ran, so this is a [3, 224, 224] tensor
img, label = dataset[0]
print("Sample image tensor shape:", img.shape, "| its class index:", label.item())

Classes (alphabetical order): ['defective', 'good']
Class to index: {'defective': 0, 'good': 1}
Total images found: 1856
Smoke-test images per class index: {0: 150, 1: 150}


AttributeError: 'int' object has no attribute 'item'

In [5]:
# Fixed random seed -> the SAME 80/20 split every time the notebook is run
generator = torch.Generator().manual_seed(42)

# 80% for training, 20% for validation
train_ds, val_ds = random_split(dataset, [0.8, 0.2], generator=generator)

print("Train images:  ", len(train_ds))
print("Validation:    ", len(val_ds))

Train images:   240
Validation:     60


In [6]:
# Default to CPU; upgrade to a GPU if one is available on this machine
device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")      # NVIDIA GPU
elif torch.backends.mps.is_available():
    device = torch.device("mps")       # Apple GPU
print("Device:", device)

Device: cpu


In [7]:
# Batched iteration over the train and validation sets
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False)

# Check that the data arrives with the correct shapes
for X, y in train_loader:
    print("X shape:", X.shape)   # torch.Size([4, 3, 224, 224])
    print("y shape:", y.shape)   # torch.Size([4])
    break                        # one batch is enough to verify

X shape: torch.Size([4, 3, 224, 224])
y shape: torch.Size([4])


In [8]:
# Build ResNet50 with its pretrained ImageNet weights
resnet50 = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# This is the layer we are going to drop (the 1000-class ImageNet head):
print("Original last layer:\n", resnet50.fc)
# -> Linear(in_features=2048, out_features=1000, bias=True)

# Replace the 1000-class head with Identity (returns the input unchanged).
# ResNet50 is now a feature extractor: image -> 2048-dim feature vector.
resnet50.fc = nn.Identity()

Original last layer:
 Linear(in_features=2048, out_features=1000, bias=True)


In [9]:
# Freeze the backbone: no gradient tracking for any ResNet50 parameter -> faster + safer
for param in resnet50.parameters():
    param.requires_grad = False

resnet50 = resnet50.to(device)
resnet50.eval()   # BatchNorm uses running stats, dropout off -> stable features

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [10]:
# What does ResNet50 output now that its last layer is Identity?
for X, y in train_loader:
    features = resnet50(X.to(device))
    print("Feature vector shape:", features.shape)   # torch.Size([4, 2048])
    print("Label shape:         ", y.shape)          # torch.Size([4])
    break

Feature vector shape: torch.Size([4, 2048])
Label shape:          torch.Size([4])


In [11]:
# Part 2 of transfer learning: the TRAINABLE part of the model (our new head)
head = nn.Sequential(
    nn.Linear(2048, 1024),   # 2048 backbone features -> 1024 learned combinations
    nn.ReLU(),               # non-linearity
    nn.Linear(1024, 1)       # 1 output neuron: 1 sigmoid value (1 = good, 0 = defective)
)
head = head.to(device)

# Full model = frozen ResNet50 (feature extractor) + trainable head
model = nn.Sequential(resnet50, head)

# Check: what shape does the FULL model now produce?
for X, y in train_loader:
    print("Model output shape:", model(X.to(device)).shape)   # torch.Size([4, 1])
    break

Model output shape: torch.Size([4, 1])


In [12]:
# How many parameters are we actually learning?
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:     {total_params:,}")      # ~27.6 million
print(f"Trainable parameters: {trainable_params:,}")  # ~2.1 million (only our head)
print(f"Trainable fraction:   {100 * trainable_params / total_params:.1f}%")

Total parameters:     25,607,233
Trainable parameters: 2,099,201
Trainable fraction:   8.2%


In [13]:
# 1. Target specifically layer4 and unfreeze its weights
for param in resnet50.layer4.parameters():
    param.requires_grad = True

# 2. Verify the change by recounting the trainable parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters (Layer 4 + FC Head): {trainable_params:,}")
print(f"Trainable fraction: {100 * trainable_params / total_params:.1f}%")

Total parameters: 25,607,233
Trainable parameters (Layer 4 + FC Head): 17,063,937
Trainable fraction: 66.6%


In [15]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), 
    lr=1e-4
)

In [16]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for X, y in dataloader:
        X, y = X.to(device), y.to(device).float().unsqueeze(1)
        
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * X.size(0)
        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct += (preds == y).sum().item()
        total += y.size(0)
        
    return running_loss / total, correct / total

In [17]:
epochs = 5

for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    print(f"Epoch {epoch+1}/{epochs} -> Loss: {train_loss:.4f} | Acc: {train_acc*100:.2f}%")

Epoch 1/5 -> Loss: 0.5604 | Acc: 73.75%
Epoch 2/5 -> Loss: 0.2615 | Acc: 91.25%
Epoch 3/5 -> Loss: 0.1796 | Acc: 93.75%
Epoch 4/5 -> Loss: 0.1495 | Acc: 96.25%
Epoch 5/5 -> Loss: 0.1023 | Acc: 97.08%
